# Exploring the Dataset: Intranet Server Audit Log

**Goal:** Explore `intranet_server/logs/audit/audit.log` (Linux auditd) and define a parsed staging model (`stg_audit_line_raw`) for later schema finalization.

This notebook walks through:
1. Loading the raw audit log (2,316 lines of `type=X msg=audit(epoch:serial): key=value` format)
2. Parsing the two-level structure (outer key-value pairs + nested `msg='...'` block)
3. Exploring every field at every nesting level
4. Building a parsed staging DataFrame
5. Integrating ground truth labels (9 labeled lines, privilege escalation)
6. Mapping to planned SQL schema with both PostgreSQL and MySQL types
7. Checking for 1NF, 2NF, and 3NF violations

---

**Dataset:** AIT Log Data Set V2.0-russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064  
**Host:** intranet_server (main attack target, WordPress/intranet)  
**Linear issue:** DAT-42

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
  data-201-security-log-analysis/   <-- this repo
    notebooks/                       <-- this notebook is here
  russellmitchell/                   <-- dataset is here
```

In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"

AUDIT_LOG = DATASET_ROOT / "gather" / "intranet_server" / "logs" / "audit" / "audit.log"
LABEL_FILE = DATASET_ROOT / "labels" / "intranet_server" / "logs" / "audit" / "audit.log"

for p, name in [
    (DATASET_ROOT, "Dataset root"),
    (AUDIT_LOG, "Audit log"),
    (LABEL_FILE, "Label file"),
]:
    status = "FOUND" if p.exists() else "MISSING"
    print(f"{name}: {p.resolve()} [{status}]")

Dataset root: /Users/namansudan/Desktop/sjsu/classes/data-201/russellmitchell [FOUND]
Audit log: /Users/namansudan/Desktop/sjsu/classes/data-201/russellmitchell/gather/intranet_server/logs/audit/audit.log [FOUND]
Label file: /Users/namansudan/Desktop/sjsu/classes/data-201/russellmitchell/labels/intranet_server/logs/audit/audit.log [FOUND]


## 1. Load Raw Data

The audit log uses the Linux auditd format. Each line has the structure:
```
type=EVENT_TYPE msg=audit(EPOCH.MS:SERIAL): key=value key=value [msg='key=value key=value']
```

Two nesting levels:
- **Outer level:** `type`, `msg=audit(...)`, `pid`, `uid`, `auid`, `ses`, and sometimes type-specific fields
- **Inner level (nested msg):** Some event types have a `msg='...'` block containing `op`, `acct`, `exe`, `hostname`, `addr`, `terminal`, `res`, and others

In [2]:
# Load all lines with 1-based line numbers (matching label file convention)
with open(AUDIT_LOG) as f:
    raw_lines = f.readlines()

print(f"Total lines: {len(raw_lines)}")
print()
print("First 5 lines:")
for i, line in enumerate(raw_lines[:5], 1):
    print(f"  [{i}] {line.rstrip()}")
print()
print("Last 3 lines:")
for i, line in enumerate(raw_lines[-3:], len(raw_lines) - 2):
    print(f"  [{i}] {line.rstrip()}")

Total lines: 2316

First 5 lines:
  [1] type=USER_ACCT msg=audit(1642723741.072:375): pid=10125 uid=0 auid=4294967295 ses=4294967295 msg='op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'
  [2] type=CRED_ACQ msg=audit(1642723741.072:376): pid=10125 uid=0 auid=4294967295 ses=4294967295 msg='op=PAM:setcred acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'
  [3] type=LOGIN msg=audit(1642723741.076:377): pid=10125 uid=0 old-auid=4294967295 auid=0 tty=(none) old-ses=4294967295 ses=65 res=1
  [4] type=USER_START msg=audit(1642723741.080:378): pid=10125 uid=0 auid=0 ses=65 msg='op=PAM:session_open acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'
  [5] type=CRED_DISP msg=audit(1642723741.084:379): pid=10125 uid=0 auid=0 ses=65 msg='op=PAM:setcred acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'

Last 3 lines:
  [2314] type=USER_END msg=audit(1643067541.366:2680): 

## 2. Parse the Audit Log Format

### 2.1 Event type inventory

First, extract all distinct `type=` values and their counts.

In [3]:
import re
from collections import Counter

type_counts = Counter()
for line in raw_lines:
    m = re.match(r"type=(\S+)", line)
    if m:
        type_counts[m.group(1)] += 1

print(f"Distinct event types: {len(type_counts)}")
print()
for event_type, count in type_counts.most_common():
    print(f"  {event_type:20s} {count:5d} ({count / len(raw_lines) * 100:5.1f}%)")

Distinct event types: 15

  CRED_ACQ               308 ( 13.3%)
  USER_START             306 ( 13.2%)
  USER_ACCT              305 ( 13.2%)
  LOGIN                  304 ( 13.1%)
  CRED_DISP              302 ( 13.0%)
  USER_END               302 ( 13.0%)
  SERVICE_START          241 ( 10.4%)
  SERVICE_STOP           230 (  9.9%)
  AVC                      4 (  0.2%)
  SYSCALL                  4 (  0.2%)
  PROCTITLE                4 (  0.2%)
  USER_LOGIN               3 (  0.1%)
  USER_AUTH                1 (  0.0%)
  USER_CMD                 1 (  0.0%)
  CRED_REFR                1 (  0.0%)


### 2.2 Format categories

The 15 event types fall into distinct format categories based on their field structure.
Let's examine a sample from each category.

In [4]:
# Group lines by type and show one example of each
type_examples = {}
for line in raw_lines:
    m = re.match(r"type=(\S+)", line)
    if m and m.group(1) not in type_examples:
        type_examples[m.group(1)] = line.rstrip()

for event_type in [
    "USER_ACCT",
    "CRED_ACQ",
    "LOGIN",
    "USER_START",
    "USER_END",
    "CRED_DISP",
    "SERVICE_START",
    "SERVICE_STOP",
    "USER_AUTH",
    "USER_CMD",
    "CRED_REFR",
    "USER_LOGIN",
    "AVC",
    "SYSCALL",
    "PROCTITLE",
]:
    if event_type in type_examples:
        print(f"--- {event_type} ---")
        print(f"  {type_examples[event_type]}")
        print()

--- USER_ACCT ---
  type=USER_ACCT msg=audit(1642723741.072:375): pid=10125 uid=0 auid=4294967295 ses=4294967295 msg='op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'

--- CRED_ACQ ---
  type=CRED_ACQ msg=audit(1642723741.072:376): pid=10125 uid=0 auid=4294967295 ses=4294967295 msg='op=PAM:setcred acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'

--- LOGIN ---
  type=LOGIN msg=audit(1642723741.076:377): pid=10125 uid=0 old-auid=4294967295 auid=0 tty=(none) old-ses=4294967295 ses=65 res=1

--- USER_START ---
  type=USER_START msg=audit(1642723741.080:378): pid=10125 uid=0 auid=0 ses=65 msg='op=PAM:session_open acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'

--- USER_END ---
  type=USER_END msg=audit(1642723741.084:380): pid=10125 uid=0 auid=0 ses=65 msg='op=PAM:session_close acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'

--- CRED_DISP ---
  type=CRED

### 2.3 Parsing strategy

Based on the examples above, there are 4 format categories:

| Category | Event types | Outer fields | Has nested msg? | Notes |
|----------|-------------|-------------|-----------------|-------|
| PAM events | USER_ACCT, CRED_ACQ, USER_START, USER_END, CRED_DISP, USER_AUTH, CRED_REFR | type, audit timestamp, pid, uid, auid, ses | Yes: op, acct, exe, hostname, addr, terminal, res | Most common (88%) |
| LOGIN | LOGIN | type, audit timestamp, pid, uid, old-auid, auid, tty, old-ses, ses, res | No | Direct fields, no nested msg |
| SERVICE | SERVICE_START, SERVICE_STOP | type, audit timestamp, pid, uid, auid, ses | Yes: unit, comm, exe, hostname, addr, terminal, res | systemd service events |
| USER_CMD | USER_CMD | type, audit timestamp, pid, uid, auid, ses | Yes: cwd, cmd (hex), terminal, res | Sudo command execution |
| USER_LOGIN | USER_LOGIN | type, audit timestamp, pid, uid, auid, ses | Yes: op, id, exe, hostname, addr, terminal, res | SSH login events |
| SYSCALL | SYSCALL | type, audit timestamp, arch, syscall, success, exit, args, ppid, pid, uid, gid, etc. | No | Kernel syscall records |
| AVC | AVC | type, audit timestamp, apparmor, operation, info, profile, name, pid, comm | No | AppArmor MAC events |
| PROCTITLE | PROCTITLE | type, audit timestamp, proctitle (hex) | No | Hex-encoded command line |

All lines share the common prefix: `type=X msg=audit(EPOCH.MS:SERIAL):` giving us a timestamp and serial number.

In [5]:
from datetime import UTC, datetime

import pandas as pd


def parse_audit_line(line_num, line):
    """Parse a single audit log line into a flat dictionary.

    Extracts:
    - Common fields: line_number, type, epoch timestamp, serial
    - Outer key-value pairs (pid, uid, auid, ses, etc.)
    - The raw msg='...' string as-is (for the raw table)
    - Nested msg='...' key-value pairs as msg_* columns (for analysis)
    """
    record = {"line_number": line_num}
    line = line.rstrip()
    record["raw_line"] = line

    # Extract type
    m_type = re.match(r"type=(\S+)", line)
    if not m_type:
        return record
    record["type"] = m_type.group(1)

    # Extract audit timestamp and serial: msg=audit(EPOCH.MS:SERIAL)
    m_ts = re.search(r"msg=audit\((\d+\.\d+):(\d+)\)", line)
    if m_ts:
        epoch = float(m_ts.group(1))
        record["epoch"] = epoch
        record["serial"] = int(m_ts.group(2))
        record["timestamp"] = datetime.fromtimestamp(epoch, tz=UTC)

    # Extract nested msg='...' content first (so we can remove it before parsing outer fields)
    nested_msg = None
    m_nested = re.search(r"msg='([^']+)'", line)
    if m_nested:
        nested_msg = m_nested.group(1)
        # Store the raw msg string for the raw table
        record["msg"] = nested_msg
        # Parse nested key=value pairs for analysis columns
        # Values can be quoted ("root") or unquoted
        for m in re.finditer(r'(\w+)="?([^"\'\s]+)"?', nested_msg):
            key = f"msg_{m.group(1)}"
            record[key] = m.group(2)

    # Parse outer key=value pairs (everything after the audit(...): prefix, excluding nested msg)
    # Remove the type= and msg=audit(...): prefix
    outer = re.sub(r"^type=\S+\s+msg=audit\([^)]+\):\s*", "", line)
    # Remove the nested msg='...' if present
    if m_nested:
        outer = outer.replace(f"msg='{m_nested.group(1)}'", "")

    # Parse remaining outer key=value pairs
    for m in re.finditer(r'([\w-]+)=("[^"]*"|\([^)]*\)|\S+)', outer):
        key = m.group(1).replace("-", "_")
        val = m.group(2).strip('"')
        if key not in record:
            record[key] = val

    return record


# Parse all lines
parsed = [parse_audit_line(i, line) for i, line in enumerate(raw_lines, 1)]
print(f"Parsed {len(parsed)} records")
print(f"Sample (line 1): {parsed[0]}")

Parsed 2316 records
Sample (line 1): {'line_number': 1, 'raw_line': 'type=USER_ACCT msg=audit(1642723741.072:375): pid=10125 uid=0 auid=4294967295 ses=4294967295 msg=\'op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success\'', 'type': 'USER_ACCT', 'epoch': 1642723741.072, 'serial': 375, 'timestamp': datetime.datetime(2022, 1, 21, 0, 9, 1, 72000, tzinfo=datetime.timezone.utc), 'msg': 'op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success', 'msg_op': 'PAM:accounting', 'msg_acct': 'root', 'msg_exe': '/usr/sbin/cron', 'msg_hostname': '?', 'msg_addr': '?', 'msg_terminal': 'cron', 'msg_res': 'success', 'pid': '10125', 'uid': '0', 'auid': '4294967295', 'ses': '4294967295'}


In [6]:
df = pd.DataFrame(parsed)

# Convert numeric fields to proper types.
# In the raw log: pid, uid, gid, euid, suid, fsuid, egid, sgid, fsgid, ppid, items,
# msg_id, syscall are integers. auid, ses, old_auid, old_ses, exit can exceed INT range
# (4294967295 = 0xFFFFFFFF). a0-a3 are hex addresses so keep as strings. arch is hex.

int_cols = [
    "pid",
    "uid",
    "gid",
    "euid",
    "suid",
    "fsuid",
    "egid",
    "sgid",
    "fsgid",
    "ppid",
    "items",
    "msg_id",
    "syscall",
]
bigint_cols = ["auid", "ses", "old_auid", "old_ses", "exit"]

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

for col in bigint_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

print(f"Shape: {df.shape}")
print(f"Columns ({len(df.columns)}):")
for col in df.columns:
    non_null = df[col].notna().sum()
    nunique = df[col].nunique()
    dtype = df[col].dtype
    print(f"  {col:30s}  non-null: {non_null:5d}/{len(df)}  unique: {nunique:5d}  dtype: {dtype}")

Shape: (2316, 53)
Columns (53):
  line_number                     non-null:  2316/2316  unique:  2316  dtype: int64
  raw_line                        non-null:  2316/2316  unique:  2316  dtype: object
  type                            non-null:  2316/2316  unique:    15  dtype: object
  epoch                           non-null:  2316/2316  unique:  1272  dtype: float64
  serial                          non-null:  2316/2316  unique:  2308  dtype: int64
  timestamp                       non-null:  2316/2316  unique:  1272  dtype: datetime64[ns, UTC]
  msg                             non-null:  2000/2316  unique:    37  dtype: object
  msg_op                          non-null:  1528/2316  unique:     6  dtype: object
  msg_acct                        non-null:  1525/2316  unique:     2  dtype: object
  msg_exe                         non-null:  1999/2316  unique:     5  dtype: object
  msg_hostname                    non-null:  1999/2316  unique:     2  dtype: object
  msg_addr           

## 3. Field-by-Field Exploration

### 3.1 type (event type)

In [7]:
print("=== Event type distribution ===")
print()
type_dist = df["type"].value_counts()
for t, count in type_dist.items():
    print(f"  {t:20s} {count:5d} ({count / len(df) * 100:5.1f}%)")

print()
print("Format categories:")
pam_types = {
    "USER_ACCT",
    "CRED_ACQ",
    "USER_START",
    "USER_END",
    "CRED_DISP",
    "USER_AUTH",
    "CRED_REFR",
}
login_types = {"LOGIN"}
service_types = {"SERVICE_START", "SERVICE_STOP"}
user_cmd_types = {"USER_CMD"}
user_login_types = {"USER_LOGIN"}
syscall_types = {"SYSCALL"}
avc_types = {"AVC"}
proctitle_types = {"PROCTITLE"}

for cat_name, cat_set in [
    ("PAM events", pam_types),
    ("LOGIN", login_types),
    ("SERVICE", service_types),
    ("USER_CMD", user_cmd_types),
    ("USER_LOGIN", user_login_types),
    ("SYSCALL", syscall_types),
    ("AVC", avc_types),
    ("PROCTITLE", proctitle_types),
]:
    count = df[df["type"].isin(cat_set)].shape[0]
    print(f"  {cat_name:20s} {count:5d} ({count / len(df) * 100:5.1f}%)")

=== Event type distribution ===

  CRED_ACQ               308 ( 13.3%)
  USER_START             306 ( 13.2%)
  USER_ACCT              305 ( 13.2%)
  LOGIN                  304 ( 13.1%)
  CRED_DISP              302 ( 13.0%)
  USER_END               302 ( 13.0%)
  SERVICE_START          241 ( 10.4%)
  SERVICE_STOP           230 (  9.9%)
  AVC                      4 (  0.2%)
  SYSCALL                  4 (  0.2%)
  PROCTITLE                4 (  0.2%)
  USER_LOGIN               3 (  0.1%)
  USER_AUTH                1 (  0.0%)
  USER_CMD                 1 (  0.0%)
  CRED_REFR                1 (  0.0%)

Format categories:
  PAM events            1525 ( 65.8%)
  LOGIN                  304 ( 13.1%)
  SERVICE                471 ( 20.3%)
  USER_CMD                 1 (  0.0%)
  USER_LOGIN               3 (  0.1%)
  SYSCALL                  4 (  0.2%)
  AVC                      4 (  0.2%)
  PROCTITLE                4 (  0.2%)


### 3.2 Timestamp and serial

In [8]:
print("=== Timestamp range ===")
print(f"  Earliest: {df['timestamp'].min()}")
print(f"  Latest:   {df['timestamp'].max()}")
print(f"  Span:     {df['timestamp'].max() - df['timestamp'].min()}")
print()
print("=== Serial number range ===")
print(f"  Min: {df['serial'].min()}")
print(f"  Max: {df['serial'].max()}")
print(f"  Unique: {df['serial'].nunique()} (of {len(df)} rows)")
print()

# Check if serial is unique per line
serial_dupes = df["serial"].duplicated().sum()
print(f"  Duplicate serials: {serial_dupes}")
if serial_dupes > 0:
    dupe_serials = df[df["serial"].duplicated(keep=False)]["serial"].unique()[:5]
    print(f"  Example duplicate serials: {dupe_serials}")
    for s in dupe_serials[:3]:
        subset = df[df["serial"] == s][["line_number", "type", "serial"]]
        print(f"    Serial {s}:")
        for _, row in subset.iterrows():
            print(f"      line {row['line_number']}: {row['type']}")

=== Timestamp range ===
  Earliest: 2022-01-21 00:09:01.072000+00:00
  Latest:   2022-01-24 23:39:13.266000+00:00
  Span:     3 days 23:30:12.194000

=== Serial number range ===
  Min: 375
  Max: 2682
  Unique: 2308 (of 2316 rows)

  Duplicate serials: 8
  Example duplicate serials: [529 530 531 532]
    Serial 529:
      line 155: AVC
      line 156: SYSCALL
      line 157: PROCTITLE
    Serial 530:
      line 158: AVC
      line 159: SYSCALL
      line 160: PROCTITLE
    Serial 531:
      line 161: AVC
      line 162: SYSCALL
      line 163: PROCTITLE


In [9]:
# Events per day (using temporary series, not modifying df)
print("=== Events per day ===")
dates = df["timestamp"].dt.date
for date, count in dates.value_counts().sort_index().items():
    print(f"  {date}  {count:5d}")

=== Events per day ===
  2022-01-21    623
  2022-01-22    552
  2022-01-23    576
  2022-01-24    565


### 3.3 pid, uid, auid, ses (process and user identifiers)

In [10]:
for col in ["pid", "uid", "auid", "ses"]:
    print(f"=== {col} ===")
    non_null = df[col].notna().sum()
    print(f"  Present: {non_null}/{len(df)}")
    if non_null > 0:
        vals = df[col].dropna().value_counts()
        print(f"  Unique values: {len(vals)}")
        if len(vals) <= 15:
            for v, c in vals.items():
                print(f"    {v}: {c}")
        else:
            print("  Top 10:")
            for v, c in vals.head(10).items():
                print(f"    {v}: {c}")
    print()

=== pid ===
  Present: 2312/2316
  Unique values: 312
  Top 10:
    1: 471
    23662: 8
    15014: 7
    14362: 7
    24407: 6
    24860: 6
    25100: 6
    25027: 6
    25020: 6
    24944: 6

=== uid ===
  Present: 2308/2316
  Unique values: 3
    0: 2303
    33: 4
    1002: 1

=== auid ===
  Present: 2308/2316
  Unique values: 3
    0: 1192
    4294967295: 1092
    1002: 24

=== ses ===
  Present: 2308/2316
  Unique values: 306
  Top 10:
    4294967295: 1092
    111: 6
    100: 6
    270: 4
    269: 4
    268: 4
    267: 4
    266: 4
    265: 4
    264: 4



### 3.4 Nested msg fields: op, acct, exe, hostname, addr, terminal, res

In [11]:
nested_fields = [
    "msg_op",
    "msg_acct",
    "msg_exe",
    "msg_hostname",
    "msg_addr",
    "msg_terminal",
    "msg_res",
]

for col in nested_fields:
    if col not in df.columns:
        print(f"=== {col} === NOT PRESENT")
        print()
        continue
    print(f"=== {col} ===")
    non_null = df[col].notna().sum()
    print(f"  Present: {non_null}/{len(df)} ({non_null / len(df) * 100:.1f}%)")
    vals = df[col].dropna().value_counts()
    print(f"  Unique values: {len(vals)}")
    if len(vals) <= 20:
        for v, c in vals.items():
            print(f"    {v}: {c}")
    else:
        print("  Top 10:")
        for v, c in vals.head(10).items():
            print(f"    {v}: {c}")
    print()

=== msg_op ===
  Present: 1528/2316 (66.0%)
  Unique values: 6
    PAM:setcred: 611
    PAM:session_open: 306
    PAM:accounting: 305
    PAM:session_close: 302
    login: 3
    PAM:authentication: 1

=== msg_acct ===
  Present: 1525/2316 (65.8%)
  Unique values: 2
    root: 1494
    jhall: 31

=== msg_exe ===
  Present: 1999/2316 (86.3%)
  Unique values: 5
    /usr/sbin/cron: 1490
    /lib/systemd/systemd: 480
    /usr/sbin/sshd: 21
    /bin/su: 4
    /usr/bin/sudo: 4

=== msg_hostname ===
  Present: 1999/2316 (86.3%)
  Unique values: 2
    ?: 1978
    172.19.131.174: 21

=== msg_addr ===
  Present: 1999/2316 (86.3%)
  Unique values: 2
    ?: 1978
    172.19.131.174: 21

=== msg_terminal ===
  Present: 2000/2316 (86.4%)
  Unique values: 6
    cron: 1490
    ?: 480
    ssh: 18
    /dev/pts/1: 8
    /dev/pts/0: 3
    pts/1: 1

=== msg_res ===
  Present: 2000/2316 (86.4%)
  Unique values: 1
    success: 2000



### 3.5 Fields by event type

Different event types populate different fields. Map which fields are present per type.

In [12]:
# For each event type, show which columns have data
print("=== Fields populated per event type ===")
print()
for event_type in df["type"].unique():
    subset = df[df["type"] == event_type]
    populated = [col for col in df.columns if subset[col].notna().any()]
    print(f"{event_type} ({len(subset)} rows):")
    print(f"  Fields: {', '.join(populated)}")
    print()

=== Fields populated per event type ===

USER_ACCT (305 rows):
  Fields: line_number, raw_line, type, epoch, serial, timestamp, msg, msg_op, msg_acct, msg_exe, msg_hostname, msg_addr, msg_terminal, msg_res, pid, uid, auid, ses

CRED_ACQ (308 rows):
  Fields: line_number, raw_line, type, epoch, serial, timestamp, msg, msg_op, msg_acct, msg_exe, msg_hostname, msg_addr, msg_terminal, msg_res, pid, uid, auid, ses

LOGIN (304 rows):
  Fields: line_number, raw_line, type, epoch, serial, timestamp, pid, uid, auid, ses, old_auid, tty, old_ses, res

USER_START (306 rows):
  Fields: line_number, raw_line, type, epoch, serial, timestamp, msg, msg_op, msg_acct, msg_exe, msg_hostname, msg_addr, msg_terminal, msg_res, pid, uid, auid, ses

CRED_DISP (302 rows):
  Fields: line_number, raw_line, type, epoch, serial, timestamp, msg, msg_op, msg_acct, msg_exe, msg_hostname, msg_addr, msg_terminal, msg_res, pid, uid, auid, ses

USER_END (302 rows):
  Fields: line_number, raw_line, type, epoch, serial, tim

### 3.6 LOGIN-specific fields: old_auid, old_ses, tty, res

In [13]:
login_df = df[df["type"] == "LOGIN"]
print(f"LOGIN events: {len(login_df)}")
print()

for col in ["old_auid", "old_ses", "tty", "res"]:
    if col in login_df.columns:
        vals = login_df[col].dropna().value_counts()
        print(f"  {col}: {dict(vals)}")
    else:
        print(f"  {col}: NOT PRESENT")
print()

# Show sample LOGIN lines
print("Sample LOGIN records:")
print(
    login_df[
        ["line_number", "pid", "uid", "old_auid", "auid", "tty", "old_ses", "ses", "res"]
    ].head()
)

LOGIN events: 304

  old_auid: {np.int64(4294967295): np.int64(304)}
  old_ses: {np.int64(4294967295): np.int64(304)}
  tty: {'(none)': np.int64(304)}
  res: {'1': np.int64(304)}

Sample LOGIN records:
    line_number    pid  uid    old_auid  auid     tty     old_ses  ses res
2             3  10125    0  4294967295     0  (none)  4294967295   65   1
10           11  10197    0  4294967295     0  (none)  4294967295   66   1
16           17  10202    0  4294967295     0  (none)  4294967295   67   1
26           27  10339    0  4294967295     0  (none)  4294967295   68   1
34           35  10409    0  4294967295     0  (none)  4294967295   69   1


### 3.7 SERVICE-specific fields: unit, comm

In [14]:
service_df = df[df["type"].isin({"SERVICE_START", "SERVICE_STOP"})]
print(f"SERVICE events: {len(service_df)}")
print()

for col in ["msg_unit", "msg_comm", "msg_exe"]:
    if col in service_df.columns:
        vals = service_df[col].dropna().value_counts()
        print(f"  {col}:")
        for v, c in vals.items():
            print(f"    {v}: {c}")
    else:
        print(f"  {col}: NOT PRESENT")
    print()

SERVICE events: 471

  msg_unit:
    phpsessionclean: 384
    apt-daily: 16
    motd-news: 16
    systemd-tmpfiles-clean: 8
    apt-daily-upgrade: 8
    user@1002: 7
    systemd-journal-flush: 3
    grub-common: 3
    systemd-journald: 3
    systemd-timesyncd: 3
    systemd-hostnamed: 3
    systemd-resolved: 3
    systemd-networkd: 3
    ssh: 3
    systemd-udevd: 3
    apport: 3
    fstrim: 2

  msg_comm:
    systemd: 471

  msg_exe:
    /lib/systemd/systemd: 471



### 3.8 USER_LOGIN-specific fields (SSH logins)

In [15]:
user_login_df = df[df["type"] == "USER_LOGIN"]
print(f"USER_LOGIN events: {len(user_login_df)}")
print()

# These have msg_op, msg_id, msg_exe, msg_hostname, msg_addr, msg_terminal, msg_res
for col in ["msg_op", "msg_id", "msg_exe", "msg_hostname", "msg_addr", "msg_terminal", "msg_res"]:
    if col in user_login_df.columns:
        vals = user_login_df[col].dropna().value_counts()
        print(f"  {col}: {dict(vals)}")
    else:
        print(f"  {col}: NOT PRESENT")

print()
print("All USER_LOGIN records:")
user_login_cols = [c for c in user_login_df.columns if user_login_df[c].notna().any()]
print(user_login_df[user_login_cols].to_string())

USER_LOGIN events: 3

  msg_op: {'login': np.int64(3)}
  msg_id: {np.int64(1002): np.int64(3)}
  msg_exe: {'/usr/sbin/sshd': np.int64(3)}
  msg_hostname: {'172.19.131.174': np.int64(3)}
  msg_addr: {'172.19.131.174': np.int64(3)}
  msg_terminal: {'/dev/pts/0': np.int64(3)}
  msg_res: {'success': np.int64(3)}

All USER_LOGIN records:
      line_number                                                                                                                                                                                                   raw_line        type         epoch  serial                        timestamp                                                                                                                msg msg_op         msg_exe    msg_hostname        msg_addr msg_terminal  msg_res    pid  uid  auid  ses  msg_id
316           317   type=USER_LOGIN msg=audit(1642761574.840:683): pid=14362 uid=0 auid=1002 ses=100 msg='op=login id=1002 exe="/usr/sbin/sshd" hostname=1

### 3.9 USER_AUTH and USER_CMD in labeled escalation lines

These are the privilege escalation events. USER_AUTH is `su` authentication, USER_CMD is `sudo` command execution.

In [16]:
# USER_AUTH
auth_df = df[df["type"] == "USER_AUTH"]
print(f"USER_AUTH events: {len(auth_df)}")
if len(auth_df) > 0:
    auth_cols = [c for c in auth_df.columns if auth_df[c].notna().any()]
    print(auth_df[auth_cols].to_string())
print()

# USER_CMD
cmd_df = df[df["type"] == "USER_CMD"]
print(f"USER_CMD events: {len(cmd_df)}")
if len(cmd_df) > 0:
    cmd_cols = [c for c in cmd_df.columns if cmd_df[c].notna().any()]
    print(cmd_df[cmd_cols].to_string())

# Decode the hex cmd field if present
if "msg_cmd" in df.columns:
    print()
    print("=== Decoded cmd (hex -> ASCII) ===")
    for _, row in cmd_df.iterrows():
        if pd.notna(row.get("msg_cmd")):
            try:
                decoded = bytes.fromhex(row["msg_cmd"]).decode("utf-8", errors="replace")
                print(f"  line {row['line_number']}: {decoded}")
            except ValueError:
                print(f"  line {row['line_number']}: [decode failed] {row['msg_cmd']}")

# CRED_REFR
refr_df = df[df["type"] == "CRED_REFR"]
print()
print(f"CRED_REFR events: {len(refr_df)}")
if len(refr_df) > 0:
    refr_cols = [c for c in refr_df.columns if refr_df[c].notna().any()]
    print(refr_df[refr_cols].to_string())

USER_AUTH events: 1
      line_number                                                                                                                                                                                                 raw_line       type         epoch  serial                        timestamp                                                                                                 msg              msg_op msg_acct  msg_exe msg_hostname msg_addr msg_terminal  msg_res    pid  uid        auid         ses
1859         1860  type=USER_AUTH msg=audit(1642999060.603:2226): pid=27950 uid=33 auid=4294967295 ses=4294967295 msg='op=PAM:authentication acct="jhall" exe="/bin/su" hostname=? addr=? terminal=/dev/pts/1 res=success'  USER_AUTH  1.642999e+09    2226 2022-01-24 04:37:40.603000+00:00  op=PAM:authentication acct="jhall" exe="/bin/su" hostname=? addr=? terminal=/dev/pts/1 res=success  PAM:authentication    jhall  /bin/su            ?        ?   /dev/pts/1  success  27950   3

### 3.10 AVC, SYSCALL, PROCTITLE (kernel-level events)

In [17]:
# AVC events
avc_df = df[df["type"] == "AVC"]
print(f"AVC events: {len(avc_df)}")
if len(avc_df) > 0:
    avc_cols = [c for c in avc_df.columns if avc_df[c].notna().any()]
    print(avc_df[avc_cols].to_string())
print()

# SYSCALL events
syscall_df = df[df["type"] == "SYSCALL"]
print(f"SYSCALL events: {len(syscall_df)}")
if len(syscall_df) > 0:
    syscall_cols = [c for c in syscall_df.columns if syscall_df[c].notna().any()]
    print(syscall_df[syscall_cols].to_string())
print()

# PROCTITLE events
proctitle_df = df[df["type"] == "PROCTITLE"]
print(f"PROCTITLE events: {len(proctitle_df)}")
if len(proctitle_df) > 0:
    proctitle_cols = [c for c in proctitle_df.columns if proctitle_df[c].notna().any()]
    print(proctitle_df[proctitle_cols].to_string())

# Decode proctitle hex
if "proctitle" in df.columns:
    print()
    print("=== Decoded proctitle (hex -> ASCII, NUL -> space) ===")
    for _, row in proctitle_df.iterrows():
        if pd.notna(row.get("proctitle")):
            try:
                decoded = (
                    bytes.fromhex(row["proctitle"])
                    .decode("utf-8", errors="replace")
                    .replace("\x00", " ")
                )
                print(f"  line {row['line_number']}: {decoded}")
            except ValueError:
                print(f"  line {row['line_number']}: [decode failed]")

AVC events: 4
     line_number                                                                                                                                                                                                                                   raw_line type         epoch  serial                        timestamp    pid apparmor        operation                               info     profile                                           name             comm
154          155                                 type=AVC msg=audit(1642746582.129:529): apparmor="STATUS" operation="profile_replace" info="same as current profile, skipping" profile="unconfined" name="/sbin/dhclient" pid=23662 comm="apparmor_parser"  AVC  1.642747e+09     529 2022-01-21 06:29:42.129000+00:00  23662   STATUS  profile_replace  same as current profile, skipping  unconfined                                 /sbin/dhclient  apparmor_parser
157          158  type=AVC msg=audit(1642746582.133:530): apparmor="STATUS

### 3.11 Serial number sharing (multi-line audit events)

Audit events with the same serial number belong to the same logical event. Check if AVC, SYSCALL, and PROCTITLE share serials.

In [18]:
# Check how many serial numbers have multiple lines
serial_counts = df.groupby("serial").size()
multi_line = serial_counts[serial_counts > 1]
print(f"Serials with >1 line: {len(multi_line)} out of {len(serial_counts)} unique serials")
print("Distribution of lines per serial:")
for n, count in serial_counts.value_counts().sort_index().items():
    print(f"  {n} line(s): {count} serials")

print()
print("Multi-line serial examples:")
for serial_num in multi_line.index[:4]:
    subset = df[df["serial"] == serial_num][["line_number", "type", "serial"]]
    print(f"  Serial {serial_num}:")
    for _, row in subset.iterrows():
        print(f"    line {row['line_number']}: {row['type']}")

Serials with >1 line: 4 out of 2308 unique serials
Distribution of lines per serial:
  1 line(s): 2304 serials
  3 line(s): 4 serials

Multi-line serial examples:
  Serial 529:
    line 155: AVC
    line 156: SYSCALL
    line 157: PROCTITLE
  Serial 530:
    line 158: AVC
    line 159: SYSCALL
    line 160: PROCTITLE
  Serial 531:
    line 161: AVC
    line 162: SYSCALL
    line 163: PROCTITLE
  Serial 532:
    line 164: AVC
    line 165: SYSCALL
    line 166: PROCTITLE


### 3.12 Session analysis

The `ses` field identifies audit sessions. Explore how sessions group events.

In [19]:
print("=== Session (ses) distribution ===")
ses_vals = df["ses"].dropna().value_counts()
print(f"Unique sessions: {len(ses_vals)}")
print()

# 4294967295 = 0xFFFFFFFF = unset/unknown
unset_ses = df[df["ses"] == 4294967295].shape[0]
print(
    f"Unset session (4294967295 = 0xFFFFFFFF): {unset_ses} rows ({unset_ses / len(df) * 100:.1f}%)"
)
print()

# Distribution of named sessions
named_ses = df[df["ses"] != 4294967295]["ses"].dropna()
print(f"Named sessions: {named_ses.nunique()} sessions, {len(named_ses)} rows")
print()

# Events per session
ses_group = df[df["ses"] != 4294967295].groupby("ses").size()
if len(ses_group) > 0:
    print(
        f"Events per named session: min={ses_group.min()}, max={ses_group.max()}, median={ses_group.median():.0f}"
    )
    print()
    print("Top 10 sessions by event count:")
    for ses_id, count in ses_group.nlargest(10).items():
        types = df[df["ses"] == ses_id]["type"].value_counts().to_dict()
        print(f"  ses={ses_id}: {count} events, types={types}")

=== Session (ses) distribution ===
Unique sessions: 306

Unset session (4294967295 = 0xFFFFFFFF): 1092 rows (47.2%)

Named sessions: 305 sessions, 1216 rows

Events per named session: min=2, max=6, median=4

Top 10 sessions by event count:
  ses=100: 6 events, types={'LOGIN': 1, 'USER_START': 1, 'CRED_ACQ': 1, 'USER_LOGIN': 1, 'USER_END': 1, 'CRED_DISP': 1}
  ses=111: 6 events, types={'LOGIN': 1, 'USER_START': 1, 'CRED_ACQ': 1, 'USER_LOGIN': 1, 'USER_END': 1, 'CRED_DISP': 1}
  ses=65: 4 events, types={'LOGIN': 1, 'USER_START': 1, 'CRED_DISP': 1, 'USER_END': 1}
  ses=66: 4 events, types={'LOGIN': 1, 'USER_START': 1, 'CRED_DISP': 1, 'USER_END': 1}
  ses=67: 4 events, types={'LOGIN': 1, 'USER_START': 1, 'CRED_DISP': 1, 'USER_END': 1}
  ses=68: 4 events, types={'LOGIN': 1, 'USER_START': 1, 'CRED_DISP': 1, 'USER_END': 1}
  ses=69: 4 events, types={'LOGIN': 1, 'USER_START': 1, 'CRED_DISP': 1, 'USER_END': 1}
  ses=70: 4 events, types={'LOGIN': 1, 'USER_START': 1, 'CRED_DISP': 1, 'USER_END': 1

### 3.13 Account (acct) analysis

The `msg_acct` field identifies which user account is referenced in each event.

In [20]:
if "msg_acct" in df.columns:
    print("=== Account (msg_acct) distribution ===")
    acct_vals = df["msg_acct"].dropna().value_counts()
    for acct, count in acct_vals.items():
        print(f"  {acct}: {count} ({count / len(df) * 100:.1f}%)")

    print()
    # Cross-tab acct vs type
    print("=== Account x Event type ===")
    ct = pd.crosstab(df["msg_acct"].fillna("[no acct]"), df["type"])
    print(ct.to_string())
else:
    print("msg_acct column not present")

=== Account (msg_acct) distribution ===
  root: 1494 (64.5%)
  jhall: 31 (1.3%)

=== Account x Event type ===
type       AVC  CRED_ACQ  CRED_DISP  CRED_REFR  LOGIN  PROCTITLE  SERVICE_START  SERVICE_STOP  SYSCALL  USER_ACCT  USER_AUTH  USER_CMD  USER_END  USER_LOGIN  USER_START
msg_acct                                                                                                                                                                
[no acct]    4         0          0          0    304          4            241           230        4          0          0         1         0           3           0
jhall        0        10          3          0      0          0              0             0        0          7          1         0         3           0           7
root         0       298        299          1      0          0              0             0        0        298          0         0       299           0         299


### 3.14 Executable (exe) analysis

In [21]:
if "msg_exe" in df.columns:
    print("=== Executable (msg_exe) distribution ===")
    exe_vals = df["msg_exe"].dropna().value_counts()
    for exe, count in exe_vals.items():
        print(f"  {exe}: {count} ({count / len(df) * 100:.1f}%)")

    print()
    # Cross-tab exe vs type
    print("=== Executable x Event type ===")
    ct = pd.crosstab(df["msg_exe"].fillna("[no exe]"), df["type"])
    print(ct.to_string())
else:
    print("msg_exe column not present")

=== Executable (msg_exe) distribution ===
  /usr/sbin/cron: 1490 (64.3%)
  /lib/systemd/systemd: 480 (20.7%)
  /usr/sbin/sshd: 21 (0.9%)
  /bin/su: 4 (0.2%)
  /usr/bin/sudo: 4 (0.2%)

=== Executable x Event type ===
type                  AVC  CRED_ACQ  CRED_DISP  CRED_REFR  LOGIN  PROCTITLE  SERVICE_START  SERVICE_STOP  SYSCALL  USER_ACCT  USER_AUTH  USER_CMD  USER_END  USER_LOGIN  USER_START
msg_exe                                                                                                                                                                            
/bin/su                 0         1          0          0      0          0              0             0        0          1          1         0         0           0           1
/lib/systemd/systemd    0         3          0          0      0          0            241           230        0          3          0         0         0           0           3
/usr/bin/sudo           0         0          1          1      0

### 3.15 Result (res) analysis

In [22]:
# Both outer res (LOGIN) and nested msg_res
print("=== Result fields ===")
for col in ["res", "msg_res"]:
    if col in df.columns:
        vals = df[col].dropna().value_counts()
        print(f"  {col}: {dict(vals)}")
    else:
        print(f"  {col}: NOT PRESENT")

# Are there any failures?
print()
for col in ["res", "msg_res"]:
    if col in df.columns:
        failures = df[~df[col].isin(["success", "1", None]) & df[col].notna()]
        print(f"  Non-success in {col}: {len(failures)} rows")
        if len(failures) > 0:
            print(failures[["line_number", "type", col]].head(10).to_string())

=== Result fields ===
  res: {'1': np.int64(304)}
  msg_res: {'success': np.int64(2000)}

  Non-success in res: 0 rows
  Non-success in msg_res: 0 rows


### 3.16 Hostname and addr fields (network context)

In [23]:
for col in ["msg_hostname", "msg_addr"]:
    if col in df.columns:
        vals = df[col].dropna().value_counts()
        print(f"=== {col} ===")
        for v, c in vals.items():
            print(f"  {v}: {c}")
        print()

# Non-? values are significant (indicate network login)
if "msg_addr" in df.columns:
    network_logins = df[(df["msg_addr"].notna()) & (df["msg_addr"] != "?")]
    print(f"Events with a real network address (not '?'): {len(network_logins)}")
    if len(network_logins) > 0:
        net_cols = [c for c in network_logins.columns if network_logins[c].notna().any()]
        print(network_logins[net_cols].to_string())

=== msg_hostname ===
  ?: 1978
  172.19.131.174: 21

=== msg_addr ===
  ?: 1978
  172.19.131.174: 21

Events with a real network address (not '?'): 21
      line_number                                                                                                                                                                                                                      raw_line        type         epoch  serial                        timestamp                                                                                                                          msg             msg_op msg_acct         msg_exe    msg_hostname        msg_addr msg_terminal  msg_res    pid  uid        auid         ses  msg_id
236           237                type=USER_END msg=audit(1642752320.510:603): pid=7708 uid=0 auid=1002 ses=29 msg='op=PAM:session_close acct="jhall" exe="/usr/sbin/sshd" hostname=172.19.131.174 addr=172.19.131.174 terminal=ssh res=success'    USER_END  1.642752e+09     603 2

## 4. Parsed Staging DataFrame

Build the definitive parsed staging DataFrame with one row per line, preserving all parsed fields.

In [24]:
# Build the raw 1:1 DataFrame for the raw table.
# The raw table stores msg as a single TEXT blob (not parsed into msg_* columns).
# The msg_* columns stay in df for analysis but are excluded from df_raw.
msg_star_cols = [c for c in df.columns if c.startswith("msg_")]
df_raw = df.drop(columns=msg_star_cols)
df_raw.insert(0, "source_host", "intranet_server")
df_raw.insert(1, "source_log", "audit.log")

print(f"Raw DataFrame shape: {df_raw.shape}")
print(f"Analysis DataFrame shape: {df.shape}")
print(f"Dropped msg_* columns: {msg_star_cols}")
print(f"Raw columns ({len(df_raw.columns)}): {list(df_raw.columns)}")
print()
print("Data types:")
print(df_raw.dtypes.to_string())
print()
print("Null counts:")
nulls = df_raw.isnull().sum()
for col, n in nulls.items():
    if n > 0:
        print(f"  {col}: {n} nulls ({n / len(df_raw) * 100:.1f}%)")

Raw DataFrame shape: (2316, 43)
Analysis DataFrame shape: (2316, 53)
Dropped msg_* columns: ['msg_op', 'msg_acct', 'msg_exe', 'msg_hostname', 'msg_addr', 'msg_terminal', 'msg_res', 'msg_unit', 'msg_comm', 'msg_id', 'msg_cwd', 'msg_cmd']
Raw columns (43): ['source_host', 'source_log', 'line_number', 'raw_line', 'type', 'epoch', 'serial', 'timestamp', 'msg', 'pid', 'uid', 'auid', 'ses', 'old_auid', 'tty', 'old_ses', 'res', 'apparmor', 'operation', 'info', 'profile', 'name', 'comm', 'arch', 'syscall', 'success', 'exit', 'a0', 'a1', 'a2', 'a3', 'items', 'ppid', 'gid', 'euid', 'suid', 'fsuid', 'egid', 'sgid', 'fsgid', 'exe', 'key', 'proctitle']

Data types:
source_host                 object
source_log                  object
line_number                  int64
raw_line                    object
type                        object
epoch                      float64
serial                       int64
timestamp      datetime64[ns, UTC]
msg                         object
pid                     

In [25]:
# Show head and tail
print("=== First 5 rows ===")
print(df_raw.head().to_string())
print()
print("=== Last 5 rows ===")
print(df_raw.tail().to_string())

=== First 5 rows ===
       source_host source_log  line_number                                                                                                                                                                                           raw_line        type         epoch  serial                        timestamp                                                                                               msg    pid  uid        auid         ses    old_auid     tty     old_ses  res apparmor operation info profile name comm arch  syscall success  exit   a0   a1   a2   a3  items  ppid   gid  euid  suid  fsuid  egid  sgid  fsgid  exe  key proctitle
0  intranet_server  audit.log            1  type=USER_ACCT msg=audit(1642723741.072:375): pid=10125 uid=0 auid=4294967295 ses=4294967295 msg='op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'   USER_ACCT  1.642724e+09     375 2022-01-21 00:09:01.072000+00:00    op=PAM:accounting acct="root

## 5. Label Integration

Load the ground truth labels for this audit log. Only 9 lines are labeled (all privilege escalation).

In [26]:
import json

labels = []
with open(LABEL_FILE) as f:
    for line in f:
        labels.append(json.loads(line))

print(f"Labeled lines: {len(labels)}")
print(f"Labeled line numbers: {[lbl['line'] for lbl in labels]}")
print()

for lbl in labels:
    print(f"  line {lbl['line']}: labels={lbl['labels']}, rules={list(lbl['rules'].keys())}")

Labeled lines: 9
Labeled line numbers: [1860, 1861, 1862, 1863, 1864, 1865, 1866, 1867, 1868]

  line 1860: labels=['attacker_change_user', 'escalate'], rules=['attacker_change_user', 'escalate']
  line 1861: labels=['attacker_change_user', 'escalate'], rules=['attacker_change_user', 'escalate']
  line 1862: labels=['attacker_change_user', 'escalate'], rules=['attacker_change_user', 'escalate']
  line 1863: labels=['attacker_change_user', 'escalate'], rules=['attacker_change_user', 'escalate']
  line 1864: labels=['escalated_command', 'escalated_sudo_command', 'escalate'], rules=['escalated_command', 'escalated_sudo_command', 'escalate']
  line 1865: labels=['escalated_command', 'escalated_sudo_command', 'escalate'], rules=['escalated_command', 'escalated_sudo_command', 'escalate']
  line 1866: labels=['escalated_command', 'escalated_sudo_command', 'escalate'], rules=['escalated_command', 'escalated_sudo_command', 'escalate']
  line 1867: labels=['escalated_command', 'escalated_sudo_co

In [27]:
# Cross-reference labeled lines with parsed data
labeled_lines = {lbl["line"] for lbl in labels}
df_labeled = df_raw[df_raw["line_number"].isin(labeled_lines)].copy()

print(f"Labeled records matched: {len(df_labeled)}")
print()

# Show the labeled records with their fields
labeled_cols = [c for c in df_labeled.columns if df_labeled[c].notna().any()]
print("Labeled records:")
print(df_labeled[labeled_cols].to_string())

Labeled records matched: 9

Labeled records:
          source_host source_log  line_number                                                                                                                                                                                                                                          raw_line        type         epoch  serial                        timestamp                                                                                                                                         msg    pid   uid        auid         ses
1859  intranet_server  audit.log         1860                                           type=USER_AUTH msg=audit(1642999060.603:2226): pid=27950 uid=33 auid=4294967295 ses=4294967295 msg='op=PAM:authentication acct="jhall" exe="/bin/su" hostname=? addr=? terminal=/dev/pts/1 res=success'   USER_AUTH  1.642999e+09    2226 2022-01-24 04:37:40.603000+00:00                                          op=PAM:authentication acct=

In [28]:
# Label distribution
from collections import Counter

label_counter = Counter()
for lbl in labels:
    for label_name in lbl["labels"]:
        label_counter[label_name] += 1

print("=== Label distribution ===")
for label_name, count in label_counter.most_common():
    print(f"  {label_name}: {count} lines")

print()
# Co-occurrence: labels are multi-valued (each line can have multiple labels)
print("=== Labels per line ===")
labels_per_line = Counter(len(lbl["labels"]) for lbl in labels)
for n, count in sorted(labels_per_line.items()):
    print(f"  {n} labels: {count} lines")

print()
print("=== Rule distribution ===")
rule_counter = Counter()
for lbl in labels:
    for rule_list in lbl["rules"].values():
        for rule_name in rule_list:
            rule_counter[rule_name] += 1

for rule_name, count in rule_counter.most_common():
    print(f"  {rule_name}: {count}")

=== Label distribution ===
  escalate: 9 lines
  escalated_command: 5 lines
  escalated_sudo_command: 5 lines
  attacker_change_user: 4 lines

=== Labels per line ===
  2 labels: 4 lines
  3 labels: 5 lines

=== Rule distribution ===
  attacker.escalate.audit.sudo.command.events: 15
  attacker.escalate.audit.su.login: 8
  attacker.escalate.audit.sudo.command.start: 3


In [29]:
# Show the raw log lines for labeled events
print("=== Raw log lines for labeled events ===")
for lbl in labels:
    line_num = lbl["line"]
    print(f"  [{line_num}] {raw_lines[line_num - 1].rstrip()}")
    print(f"         labels: {lbl['labels']}")
    print()

=== Raw log lines for labeled events ===
  [1860] type=USER_AUTH msg=audit(1642999060.603:2226): pid=27950 uid=33 auid=4294967295 ses=4294967295 msg='op=PAM:authentication acct="jhall" exe="/bin/su" hostname=? addr=? terminal=/dev/pts/1 res=success'
         labels: ['attacker_change_user', 'escalate']

  [1861] type=USER_ACCT msg=audit(1642999060.603:2227): pid=27950 uid=33 auid=4294967295 ses=4294967295 msg='op=PAM:accounting acct="jhall" exe="/bin/su" hostname=? addr=? terminal=/dev/pts/1 res=success'
         labels: ['attacker_change_user', 'escalate']

  [1862] type=CRED_ACQ msg=audit(1642999060.615:2228): pid=27950 uid=33 auid=4294967295 ses=4294967295 msg='op=PAM:setcred acct="jhall" exe="/bin/su" hostname=? addr=? terminal=/dev/pts/1 res=success'
         labels: ['attacker_change_user', 'escalate']

  [1863] type=USER_START msg=audit(1642999060.627:2229): pid=27950 uid=33 auid=4294967295 ses=4294967295 msg='op=PAM:session_open acct="jhall" exe="/bin/su" hostname=? addr=? term

## 6. Summary Statistics

In [30]:
print("=== Summary ===")
print(f"Total lines:             {len(df_raw)}")
print(f"Distinct event types:    {df_raw['type'].nunique()}")
print(f"Distinct serials:        {df_raw['serial'].nunique()}")
print(f"Time range:              {df_raw['timestamp'].min()} to {df_raw['timestamp'].max()}")
print(f"Labeled lines:           {len(labels)} (all privilege escalation)")
print(f"Columns in raw DataFrame: {len(df_raw.columns)}")
print()

# Columns with >50% nulls (type-specific fields)
print("Columns with >50% nulls (type-specific, sparse):")
for col in df_raw.columns:
    null_pct = df_raw[col].isnull().sum() / len(df_raw) * 100
    if null_pct > 50:
        print(f"  {col}: {null_pct:.1f}% null")

print()
# Dominant patterns (use analysis df which has msg_exe)
cron_events = df[df["msg_exe"] == "/usr/sbin/cron"].shape[0] if "msg_exe" in df.columns else 0
print(f"Cron job events:         {cron_events} ({cron_events / len(df_raw) * 100:.1f}%)")

service_events = df_raw[df_raw["type"].isin({"SERVICE_START", "SERVICE_STOP"})].shape[0]
print(f"Service start/stop:      {service_events} ({service_events / len(df_raw) * 100:.1f}%)")

labeled_count = len(labels)
print(f"Labeled lines: {labeled_count} ({labeled_count / len(df_raw) * 100:.1f}%)")

=== Summary ===
Total lines:             2316
Distinct event types:    15
Distinct serials:        2308
Time range:              2022-01-21 00:09:01.072000+00:00 to 2022-01-24 23:39:13.266000+00:00
Labeled lines:           9 (all privilege escalation)
Columns in raw DataFrame: 43

Columns with >50% nulls (type-specific, sparse):
  old_auid: 86.9% null
  tty: 86.7% null
  old_ses: 86.9% null
  res: 86.9% null
  apparmor: 99.8% null
  operation: 99.8% null
  info: 99.8% null
  profile: 99.8% null
  name: 99.8% null
  comm: 99.7% null
  arch: 99.8% null
  syscall: 99.8% null
  success: 99.8% null
  exit: 99.8% null
  a0: 99.8% null
  a1: 99.8% null
  a2: 99.8% null
  a3: 99.8% null
  items: 99.8% null
  ppid: 99.8% null
  gid: 99.8% null
  euid: 99.8% null
  suid: 99.8% null
  fsuid: 99.8% null
  egid: 99.8% null
  sgid: 99.8% null
  fsgid: 99.8% null
  exe: 99.8% null
  key: 99.8% null
  proctitle: 99.8% null

Cron job events:         1490 (64.3%)
Service start/stop:      471 (20.3%)
Lab

## 7. Schema Mapping

Map the parsed fields to the shared parsed staging table `stg_audit_line_raw`. This file maps to the shared staging table shape; `source_host` and `source_log` distinguish records from different audit sources.

### 7.0 Type inference assumptions

The raw audit log is unstructured text with no formal schema. All SQL type assignments are inferred from observed values. See `type_inference_assumptions.md` (at data-201/ root) for the shared team-level rules and per-file reasoning.

Key decisions for this file:
- Fields with only decimal digits mapped to INTEGER (pid, uid, gid, etc.) or BIGINT (auid, ses where 4294967295 exceeds INT range)
- Hex-valued fields (arch, a0-a3, proctitle) kept as VARCHAR/TEXT
- epoch as DOUBLE PRECISION; timestamp derived from epoch as TIMESTAMP WITH TIME ZONE
- The `msg` field stores the entire `msg='...'` string as a single TEXT blob. Parsing it into individual fields belongs in normalization, not raw storage.

### 7.1 Column mapping table

| # | Raw Field | Description | PostgreSQL Type | MySQL Type | Nullable | Notes |
|---|-----------|-------------|----------------|------------|----------|-------|
| 1 | line_number | 1-based line in source file | INTEGER | INT | NOT NULL | Joins to labels |
| 2 | type | Audit event type (USER_ACCT, LOGIN, etc.) | VARCHAR(20) | VARCHAR(20) | NOT NULL | 15 distinct values |
| 3 | epoch | Unix timestamp with milliseconds | DOUBLE PRECISION | DOUBLE | NOT NULL | Raw epoch for precision |
| 4 | serial | Audit serial number | INTEGER | INT | NOT NULL | Shared across multi-line events |
| 5 | timestamp | Parsed UTC datetime | TIMESTAMP WITH TIME ZONE | DATETIME | NOT NULL | Derived from epoch |
| 6 | pid | Process ID | INTEGER | INT | NULL | Present on 2312/2316 rows |
| 7 | uid | User ID of the process | INTEGER | INT | NULL | 0=root, 33=www-data, 1002=jhall |
| 8 | auid | Audit user ID (login UID) | BIGINT | BIGINT | NULL | 4294967295 = unset |
| 9 | ses | Session ID | BIGINT | BIGINT | NULL | 4294967295 = unset |
| 10 | msg | Full nested msg string as-is | TEXT | TEXT | NULL | Multi-valued (1NF violation); NULL for LOGIN, AVC, SYSCALL, PROCTITLE |
| 11 | old_auid | Previous audit UID | BIGINT | BIGINT | NULL | LOGIN only (always 4294967295) |
| 12 | old_ses | Previous session ID | BIGINT | BIGINT | NULL | LOGIN only (always 4294967295) |
| 13 | tty | Terminal (LOGIN format) | VARCHAR(30) | VARCHAR(30) | NULL | LOGIN and SYSCALL |
| 14 | res | Result (LOGIN format, 1/0) | VARCHAR(10) | VARCHAR(10) | NULL | LOGIN only (always "1") |
| 15 | apparmor | AppArmor status | VARCHAR(20) | VARCHAR(20) | NULL | AVC only (4 rows) |
| 16 | operation | AppArmor operation | VARCHAR(30) | VARCHAR(30) | NULL | AVC only |
| 17 | info | AppArmor info message | TEXT | TEXT | NULL | AVC only |
| 18 | profile | AppArmor profile | VARCHAR(50) | VARCHAR(50) | NULL | AVC only |
| 19 | name | AppArmor profile target | TEXT | TEXT | NULL | AVC only, 4 distinct paths |
| 20 | comm | Command name (outer) | VARCHAR(50) | VARCHAR(50) | NULL | SYSCALL and AVC |
| 21 | exe | Executable path (outer) | TEXT | TEXT | NULL | SYSCALL only |
| 22 | arch | Architecture hex code | VARCHAR(20) | VARCHAR(20) | NULL | SYSCALL only (c000003e = x86_64) |
| 23 | syscall | Syscall number | INTEGER | INT | NULL | SYSCALL only |
| 24 | success | Syscall success (yes/no) | VARCHAR(5) | VARCHAR(5) | NULL | SYSCALL only |
| 25 | exit | Syscall exit code | BIGINT | BIGINT | NULL | SYSCALL only |
| 26 | a0 | Syscall argument 0 | VARCHAR(20) | VARCHAR(20) | NULL | SYSCALL only, hex values |
| 27 | a1 | Syscall argument 1 | VARCHAR(20) | VARCHAR(20) | NULL | SYSCALL only, hex addresses |
| 28 | a2 | Syscall argument 2 | VARCHAR(20) | VARCHAR(20) | NULL | SYSCALL only, hex values |
| 29 | a3 | Syscall argument 3 | VARCHAR(20) | VARCHAR(20) | NULL | SYSCALL only, hex values |
| 30 | items | Number of path records | INTEGER | INT | NULL | SYSCALL only |
| 31 | ppid | Parent process ID | INTEGER | INT | NULL | SYSCALL only |
| 32 | gid | Group ID | INTEGER | INT | NULL | SYSCALL only |
| 33 | euid | Effective user ID | INTEGER | INT | NULL | SYSCALL only |
| 34 | suid | Saved user ID | INTEGER | INT | NULL | SYSCALL only |
| 35 | fsuid | Filesystem user ID | INTEGER | INT | NULL | SYSCALL only |
| 36 | egid | Effective group ID | INTEGER | INT | NULL | SYSCALL only |
| 37 | sgid | Saved group ID | INTEGER | INT | NULL | SYSCALL only |
| 38 | fsgid | Filesystem group ID | INTEGER | INT | NULL | SYSCALL only |
| 39 | key | Audit filter key | VARCHAR(20) | VARCHAR(20) | NULL | SYSCALL only (always "(null)") |
| 40 | proctitle | Hex-encoded process title | TEXT | TEXT | NULL | PROCTITLE only (4 rows) |
| 41 | source_host | Host producing this audit log | VARCHAR(30) | VARCHAR(30) | NOT NULL | Provenance: constant per source file |
| 42 | source_log | Log file within host | VARCHAR(50) | VARCHAR(50) | NOT NULL | Provenance: constant per source file |
| 43 | raw_line | Original unparsed log line | TEXT | TEXT | NOT NULL | Source-faithful anchor for debugging |


### 7.2 Parsed Staging DDL

This file maps to the shared staging table shape `stg_audit_line_raw`. The `source_host` and `source_log` columns distinguish records from different audit sources.

In [31]:
postgresql_ddl = """
-- PostgreSQL
CREATE TABLE stg_audit_line_raw (
    row_id          SERIAL PRIMARY KEY,
    source_host     VARCHAR(30) NOT NULL,
    source_log      VARCHAR(50) NOT NULL,
    line_number     INTEGER NOT NULL,
    raw_line        TEXT NOT NULL,
    type            VARCHAR(20) NOT NULL,
    epoch           DOUBLE PRECISION NOT NULL,
    serial          INTEGER NOT NULL,
    timestamp       TIMESTAMP WITH TIME ZONE NOT NULL,
    pid             INTEGER,
    uid             INTEGER,
    auid            BIGINT,
    ses             BIGINT,
    msg             TEXT,
    old_auid        BIGINT,
    old_ses         BIGINT,
    tty             VARCHAR(30),
    res             VARCHAR(10),
    apparmor        VARCHAR(20),
    operation       VARCHAR(30),
    info            TEXT,
    profile         VARCHAR(50),
    name            TEXT,
    comm            VARCHAR(50),
    exe             TEXT,
    arch            VARCHAR(20),
    syscall         INTEGER,
    success         VARCHAR(5),
    exit            BIGINT,
    a0              VARCHAR(20),
    a1              VARCHAR(20),
    a2              VARCHAR(20),
    a3              VARCHAR(20),
    items           INTEGER,
    ppid            INTEGER,
    gid             INTEGER,
    euid            INTEGER,
    suid            INTEGER,
    fsuid           INTEGER,
    egid            INTEGER,
    sgid            INTEGER,
    fsgid           INTEGER,
    key             VARCHAR(20),
    proctitle       TEXT
);
"""

mysql_ddl = """
-- MySQL
CREATE TABLE stg_audit_line_raw (
    row_id          INT AUTO_INCREMENT PRIMARY KEY,
    source_host     VARCHAR(30) NOT NULL,
    source_log      VARCHAR(50) NOT NULL,
    line_number     INT NOT NULL,
    raw_line        TEXT NOT NULL,
    type            VARCHAR(20) NOT NULL,
    epoch           DOUBLE NOT NULL,
    serial          INT NOT NULL,
    timestamp       DATETIME NOT NULL,
    pid             INT,
    uid             INT,
    auid            BIGINT,
    ses             BIGINT,
    msg             TEXT,
    old_auid        BIGINT,
    old_ses         BIGINT,
    tty             VARCHAR(30),
    res             VARCHAR(10),
    apparmor        VARCHAR(20),
    operation       VARCHAR(30),
    info            TEXT,
    profile         VARCHAR(50),
    name            TEXT,
    comm            VARCHAR(50),
    exe             TEXT,
    arch            VARCHAR(20),
    syscall         INT,
    success         VARCHAR(5),
    exit            BIGINT,
    a0              VARCHAR(20),
    a1              VARCHAR(20),
    a2              VARCHAR(20),
    a3              VARCHAR(20),
    items           INT,
    ppid            INT,
    gid             INT,
    euid            INT,
    suid            INT,
    fsuid           INT,
    egid            INT,
    sgid            INT,
    fsgid           INT,
    `key`           VARCHAR(20),
    proctitle       TEXT
);
"""

print(postgresql_ddl)
print(mysql_ddl)


-- PostgreSQL
CREATE TABLE stg_audit_line_raw (
    row_id          SERIAL PRIMARY KEY,
    source_host     VARCHAR(30) NOT NULL,
    source_log      VARCHAR(50) NOT NULL,
    line_number     INTEGER NOT NULL,
    raw_line        TEXT NOT NULL,
    type            VARCHAR(20) NOT NULL,
    epoch           DOUBLE PRECISION NOT NULL,
    serial          INTEGER NOT NULL,
    timestamp       TIMESTAMP WITH TIME ZONE NOT NULL,
    pid             INTEGER,
    uid             INTEGER,
    auid            BIGINT,
    ses             BIGINT,
    msg             TEXT,
    old_auid        BIGINT,
    old_ses         BIGINT,
    tty             VARCHAR(30),
    res             VARCHAR(10),
    apparmor        VARCHAR(20),
    operation       VARCHAR(30),
    info            TEXT,
    profile         VARCHAR(50),
    name            TEXT,
    comm            VARCHAR(50),
    exe             TEXT,
    arch            VARCHAR(20),
    syscall         INTEGER,
    success         VARCHAR(5),
    ex

## 8. Normalization Observations

Applying the `normalization_rules_sheet.md` checklist to the raw audit log data.

### 8.1 1NF Check

**Multi-valued field:** The `msg` column contains multiple key-value pairs packed into a single text field (e.g., `op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success`). This is a **1NF violation**-multiple distinct values (operation, account, executable, hostname, address, terminal, result) stored in one cell. Same pattern as `groups TEXT` and `add_field_json TEXT` in `stg_host_raw`.

The `msg` field is non-null for ~86% of rows (PAM events, SERVICE events, USER_LOGIN, USER_CMD). It is NULL for LOGIN, AVC, SYSCALL, and PROCTITLE event types, which carry their fields as top-level key-value pairs instead.

**Repeating groups:** None. No field1, field2, field3 patterns.

**1NF status: violated.** The `msg` column packs multiple key-value pairs into a single text blob. Normalization should unpack it into separate columns (op, acct, exe, hostname, addr, terminal, res, unit, comm, id, cwd, cmd).

Note: the companion label file (`labels/intranet_server/logs/audit/audit.log`) has multi-valued fields (`labels` is an array, `rules` is a nested dict), but those are part of the label dataset analyzed in DAT-48, not part of this raw table.

### 8.2 2NF Check

**Primary key:** The raw table uses a single-column surrogate primary key (`row_id`). Partial dependencies require a composite primary key (two or more columns), which does not exist here.

**2NF status: satisfied.** Single-column PK makes partial dependencies impossible.

Note for normalization phase: the `serial` field groups related lines into logical audit events (e.g., AVC + SYSCALL + PROCTITLE sharing serial 529). If (serial, type) were used as a composite key, then type-specific fields (apparmor, proctitle, syscall args, etc.) would be partial dependencies on `type` alone. This is relevant design input for the final normalized audit schema.

### 8.3 3NF Check

**Transitive dependencies identified:**

| Determinant | Dependent(s) | Pattern | Notes |
|-------------|-------------|---------|-------|
| type | set of non-null columns | Each event type determines which fields are populated. PAM events populate msg (with op/acct/exe/res inside); SYSCALL populates arch/syscall/ppid/comm/exe/a0-a3/gid/euid/etc; PROCTITLE only has proctitle. | Structural: the audit log format defines which fields exist per event type. This drives the normalization decision to split into type-specific subtables. |

**What this means:** The staging table has 43 columns, but any individual row uses at most ~15 because `type` determines the field population pattern. For example, row_id -> type -> {arch, syscall, ppid, gid, euid, ...} forms a transitive chain where the SYSCALL-specific columns depend on type (non-key), not directly on row_id. Event type strongly correlates with which fields are populated, which indicates structural heterogeneity and may motivate subtype modeling in the final design.

**3NF status: structural heterogeneity noted.** The type -> field_set correlation is a key normalization input for the final design.

**Not a 3NF violation:**
- msg_op does not determine msg_exe (when unpacked). While cron events always use `/usr/sbin/cron`, other ops (PAM:setcred) appear with multiple executables. The exe depends on the event context (acct + process), not solely on the operation type.

### 8.4 Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) | Reasoning |
|---|---|---|---|
| FD1 | row_id | all attributes | Surrogate PK, trivially determines everything. |
| FD2 | (source_host, source_log, line_number) | all attributes | Each line in a given source file is unique. Staging candidate key across files. |
| FD3 | (serial, type) | all type-specific attributes | Within a serial group, each type appears at most once. 2304 serials have 1 line, 4 serials have 3 lines (AVC+SYSCALL+PROCTITLE). Candidate composite key for a normalized multi-line event table. |
| FD4 | type | set of non-null columns | Structural FD: determines which fields have values. Drives 3NF decomposition into type-specific subtables. |


## 9. Key Findings for Schema Design

1. **Format heterogeneity:** 15 event types across 4+ format categories. The staging table has 43 columns but any individual row uses at most ~15, because `type` determines which fields are populated. This sparsity indicates structural heterogeneity and may motivate subtype modeling in the final design.

2. **1NF violation in msg column:** The `msg` TEXT column packs multiple key-value pairs (op, acct, exe, hostname, addr, terminal, res, unit, comm, id, cwd, cmd) into a single text blob. Same pattern as `groups TEXT` in stg_host_raw. Normalization should unpack these into separate columns.

3. **Serial groups multi-line events:** 4 serials (529-532) span 3 lines each (AVC + SYSCALL + PROCTITLE). The remaining 2,304 serials each have 1 line. Serial grouping is an important design input; final grain will be decided in schema finalization docs.

4. **Labeled lines are rare:** Only 9 of 2,316 lines (0.4%) have attack labels. Lines 1860-1863: `su` from www-data (uid 33) to jhall via `/bin/su` (attacker_change_user, escalate). Lines 1864-1868: `sudo cat /etc/shadow` by jhall via `/usr/bin/sudo` (escalated_command, escalated_sudo_command, escalate). The attacker IP (172.19.131.174) appears in 21 rows including 3 USER_LOGIN events (SSH logins as uid 1002 / jhall).

5. **Dominant pattern is cron:** ~64% of events are cron job PAM cycles (root, /usr/sbin/cron). Another ~20% are systemd SERVICE_START/STOP. This is the dominant unlabeled activity in this file.

6. **Unset sentinel values:** auid=4294967295 (0xFFFFFFFF) in 1,092 rows and ses=4294967295 in 1,092 rows indicate system processes (cron, systemd) without an interactive login session. These are valid values, not NULLs. Normalization should decide: store as-is, convert to NULL, or use a flag column.

7. **Dual result fields:** PAM events use msg_res ("success"/"failed" text) inside the msg blob, LOGIN uses res ("1"/"0" numeric string) as a top-level field. Normalization should unify into a single result representation.

8. **Cross-file synthesis with internal_share audit.log (DAT-36):** This file and `internal_share` share the same auditd source format and may share a common final audit schema after cross-file synthesis. The intranet_server file (2,316 rows) covers privilege escalation; the internal_share file (732 rows) covers exfiltration.

9. **Hex-encoded fields:** The msg blob in USER_CMD contains hex-encoded command data. proctitle in PROCTITLE (4 rows) contains hex-encoded command line. Normalization should decide whether to store decoded or raw hex.
